| HTTP Code                     | Meaning / When to Use in RAG API                                 | Common Error / Example                                        |
| ----------------------------- | ---------------------------------------------------------------- | ------------------------------------------------------------- |
| **200 OK**                    | Successful retrieval of documents or LLM-generated answer        | Usually none; normal response                                 |
| **201 Created**               | When a new resource is added (e.g., new document indexed)        | If POST fails to store the document → 500/422                 |
| **204 No Content**            | Successful request with no response body (e.g., delete document) | Misused if client expects data back                           |
| **400 Bad Request**           | Invalid query format, missing required parameters, wrong JSON    | Malformed query JSON, missing `query` field                   |
| **401 Unauthorized**          | Authentication missing or invalid (e.g., API key)                | Calling RAG API without token or expired key                  |
| **403 Forbidden**             | Authenticated but no permission to access resource               | User not allowed to access certain documents                  |
| **404 Not Found**             | Document, chunk, or resource does not exist                      | Querying non-existent document ID or page chunk               |
| **409 Conflict**              | Resource already exists (e.g., duplicate document)               | Attempting to index a PDF with same ID twice                  |
| **422 Unprocessable Entity**  | Input validation failed                                          | Invalid data type for query or embedding request              |
| **429 Too Many Requests**     | Rate limit exceeded                                              | Sending too many RAG requests to API in short time            |
| **500 Internal Server Error** | LLM or backend failure                                           | RAG pipeline crashes during LLM call or embedding computation |
| **503 Service Unavailable**   | Temporary backend issue / overload                               | Vector DB or LLM service down temporarily                     |


# FastAPI 

# Basics

## What is FastAPI and how is it different from Flask or Django?

FastAPI is a modern, high-performance Python web framework for building APIs with automatic OpenAPI documentation, type checking, and asynchronous support. It is designed for speed, efficiency, and developer productivity, particularly for RESTful APIs and microservices.

## What are the advantages of FastAPI for building APIs?

FastAPI is a high-performance, modern Python framework that simplifies API development with automatic validation, docs, async support, and type safety.

## How does FastAPI handle requests and responses?

FastAPI automatically receives, validates, and parses request data, passes it to your endpoint function, and returns JSON responses, supporting async execution and automatic API documentation.

# Models & Validation

## What are Pydantic models and how are they used in FastAPI?

Pydantic models are Python classes used for data validation, parsing, and serialization. They ensure that the data conforms to the expected types and structure before your application processes it.

```
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float
    in_stock: bool

@app.post("/items/")
async def create_item(item: Item):
    return {"item_name": item.name, "item_price": item.price}
```

## How do you validate request data using FastAPI?

FastAPI validates request data automatically using Pydantic models, type hints, and optional custom validators, returning clear errors for invalid input.

## How do you handle path, query, and request body parameters?

FastAPI handles path, query, and body parameters automatically using Python type hints, Query/Path utilities, and Pydantic models for parsing and validation.

```
@app.post("/users/{user_id}/items/")
async def create_user_item(
    user_id: int,                       # Path parameter
    category: str = Query(...),         # Query parameter
    item: Item                          # Body parameter
):
    return {"user_id": user_id, "category": category, "item_name": item.name}

```


# Asynchronous Programming

## What is async/await in FastAPI and why is it important?

async/await are Python keywords that enable asynchronous programming, allowing functions to run without blocking while waiting for I/O operations like API calls, database queries, or file operations.

```
from fastapi import FastAPI
import asyncio

app = FastAPI()

async def fetch_data():
    await asyncio.sleep(2)  # Simulate I/O
    return {"data": "Hello"}

@app.get("/async")
async def get_data():
    result = await fetch_data()
    return result

```

# Security

## How do you implement authentication and authorization in FastAPI?

FastAPI provides flexible tools to implement authentication (verifying identity) and authorization (controlling access) using dependencies, security utilities, and OAuth2/JWT tokens.

```
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer

app = FastAPI()
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")

def get_current_user(token: str = Depends(oauth2_scheme)):
    if token != "fake-token":
        raise HTTPException(status_code=401, detail="Invalid token")
    return {"username": "john"}

@app.get("/users/me")
async def read_users_me(current_user: dict = Depends(get_current_user)):
    return current_user

```

## How do you implement CORS in FastAPI?

CORS (Cross-Origin Resource Sharing) is a security feature that allows your API to be accessed from different domains or origins (e.g., frontend apps hosted elsewhere).

```
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

# Allowed origins (frontend domains)
origins = [
    "http://localhost:3000",
    "https://myfrontend.com"
]

# Add CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,         # Allowed domains
    allow_credentials=True,        # Allow cookies / credentials
    allow_methods=["*"],           # GET, POST, PUT, DELETE, etc.
    allow_headers=["*"],           # Allowed headers
)
```

## How do you handle sensitive data or PII in FastAPI APIs?

In FastAPI, sensitive data and PII are protected using HTTPS, authentication/authorization, encryption, input validation, and careful response modeling to prevent exposure.


# Routing & Dependency Management

## How does dependency injection work in FastAPI?

Dependency injection (DI) is a design pattern where components (dependencies) are provided to a function instead of being created inside it. In FastAPI, DI is used to inject reusable logic, services, or resources into endpoints, making code modular, testable, and maintainable.

```

from fastapi import FastAPI, Depends

app = FastAPI()

def get_db():
    db = "DB Connection"
    try:
        yield db
    finally:
        print("Close DB connection")  # cleanup

@app.get("/items/")
async def read_items(db = Depends(get_db)):
    return {"db_connection": db}

```
## How do you implement middleware in FastAPI?

Middleware in FastAPI is a function or class that runs before and/or after every request. It is used for tasks like logging, authentication, CORS handling, request modification, or error handling.

```
from starlette.middleware.base import BaseHTTPMiddleware
from fastapi import FastAPI, Request

app = FastAPI()

class CustomMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        print("Before request")
        response = await call_next(request)
        print("After request")
        return response

app.add_middleware(CustomMiddleware)

@app.get("/")
async def root():
    return {"message": "Hello from class middleware"}
```

# Background Tasks & Files

## How do you use background tasks in FastAPI?

Background tasks allow you to run operations after returning a response to the client, without making them wait. This is useful for sending emails, logging, processing files, or any non-blocking work.

```
def notify_admin(msg: str):
    print(f"Admin notified: {msg}")

@app.post("/process/")
async def process_data(data: str, background_tasks: BackgroundTasks):
    background_tasks.add_task(write_log, data)
    background_tasks.add_task(notify_admin, data)
    return {"status": "Processing started"}
```

## How do you handle file uploads and downloads?

FastAPI handles file uploads with UploadFile and File, and downloads with FileResponse or StreamingResponse, allowing efficient and async-friendly file operations.

```
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse
from fastapi.responses import StreamingResponse

app = FastAPI()

@app.post("/upload/")
async def upload_file(file: UploadFile = File(...)):
    content = await file.read()  # Read file content (async)
    with open(f"uploads/{file.filename}", "wb") as f:
        f.write(content)
    return {"filename": file.filename, "content_type": file.content_type}

```
## How do you serve static files and templates?

FastAPI allows you to serve static files (images, CSS, JS) and render HTML templates using Jinja2 or other template engines.


# Documentation & Testing

## How do you automatically generate API documentation using OpenAPI / Swagger?

FastAPI automatically generates interactive API documentation using OpenAPI, Swagger UI (/docs) and Redoc (/redoc) by reading endpoint definitions, Pydantic models, and type hints.

## How do you test FastAPI endpoints using TestClient or pytest?

FastAPI makes testing easy with Starlette’s TestClient, which allows you to simulate requests to your API endpoints without running a server. Combined with pytest, you can write automated tests for all routes.

```
import pytest

@pytest.fixture
def client_fixture():
    from fastapi.testclient import TestClient
    from main import app  # Import your FastAPI app
    return TestClient(app)

def test_hello(client_fixture):
    response = client_fixture.get("/hello")
    assert response.status_code == 200

```

# Deployment & Performance

## How do you deploy FastAPI applications (Uvicorn, Gunicorn, Docker)?

FastAPI apps are deployed using ASGI servers like Uvicorn for simple setups, Gunicorn with Uvicorn workers for production, and optionally Docker for containerized, scalable deployments.

## How do you optimize FastAPI for high performance and concurrency?

Use async endpoints (async def) for I/O-bound tasks    

Deploy with Gunicorn + Uvicorn workers for production concurrency               

Use async database drivers and connection pooling    

Implement caching (in-memory or Redis) for frequently accessed data         

Offload long tasks to background tasks or task queues
Enable Gzip / response compression      

Use efficient Pydantic models and avoid sending unnecessary data        

# Advanced / Optional Topics

## How do you handle API versioning in FastAPI?

FastAPI does not enforce a specific versioning system, but you can implement versioning using URL paths, headers, or query parameters.

```
from fastapi import FastAPI

app = FastAPI()

@app.get("/v1/items/")
async def get_items_v1():
    return {"version": "v1", "items": ["item1", "item2"]}

@app.get("/v2/items/")
async def get_items_v2():
    return {"version": "v2", "items": ["item1", "item2", "item3"]}
```

## How do you implement WebSockets for real-time communication?

In FastAPI, WebSockets provide real-time bidirectional communication, implemented via WebSocket endpoints with async receive and send methods, supporting single or multiple clients.

```
from fastapi import FastAPI, WebSocket

app = FastAPI()

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()  # Accept the connection
    while True:
        data = await websocket.receive_text()  # Receive message from client
        await websocket.send_text(f"Message received: {data}")  # Send response
        
```

## How do you run startup and shutdown event handlers?


